In [2]:
# 1. Install necessary libraries
!pip install -q transformers accelerate huggingface_hub

# 2. Imports
import os
import torch

from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

# 3. Retrieve your Hugging Face token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get('Token_Llama-3.2-1B')

# Login to Hugging Face
login(os.environ["HF_TOKEN"])

# 4. Load model and tokenizer
model_id = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map="auto"
)

model.eval()

# Optional: confirm GPU
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 5. Function to compute log P(B | A)
def log_prob(A, B):

    device = model.device

    # Tokenize prompt alone
    tokens_A = tokenizer(A, return_tensors="pt").to(device)

    # Tokenize prompt + continuation
    tokens_full = tokenizer(A + B, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**tokens_full)

    logits = outputs.logits
    log_probs = torch.log_softmax(logits, dim=-1)

    input_ids = tokens_full["input_ids"][0]

    len_A = tokens_A["input_ids"].shape[1]

    total_log_prob = 0.0

    # Sum token log probabilities for B only
    for i in range(len_A, len(input_ids)):
        token_id = input_ids[i]
        total_log_prob += log_probs[0, i - 1, token_id].item()

    return total_log_prob

# 6. Example experiment
A = "The capital of France is"

B1 = " Paris."
B2 = " London."

score1 = log_prob(A, B1)
score2 = log_prob(A, B2)

print()
print("log P(Paris | A)  =", score1)
print("log P(London | A) =", score2)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

GPU available: True
GPU: Tesla T4

log P(Paris | A)  = -1.8974609375
log P(London | A) = -9.80615234375


In [4]:
def log_prob(A, B):
    device = model.device

    # Standardize the transition
    # We ensure B starts with a space if it doesn't have one,
    # reflecting natural linguistic flow after a period.
    if not B.startswith(" "):
        B = " " + B

    # Tokenize the full sequence ONLY
    # This ensures we see how A and B interact/merge.
    full_text = A + B
    tokens_full = tokenizer(full_text, return_tensors="pt").to(device)
    input_ids = tokens_full["input_ids"][0]

    # Robust split-point detection
    # We tokenize A alone just to find its length in the context of the full string.
    # We use 'add_special_tokens=True' to ensure the BOS token is accounted for.
    tokens_A = tokenizer(A, return_tensors="pt")
    len_A = tokens_A["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model(tokens_full["input_ids"])

    # Proper Logit Shifting
    # Logits at index i predict the token at index i+1.
    logits = outputs.logits
    log_probs = torch.log_softmax(logits, dim=-1)

    total_log_prob = 0.0

    # Cleaner indexing loop
    # We start from len_A because that is the first token of "B".
    # We pull the probability from the logit at (i - 1).
    for i in range(len_A, len(input_ids)):
        token_id = input_ids[i]
        # Get the model's confidence for this token based on all previous tokens
        token_score = log_probs[0, i - 1, token_id].item()
        total_log_prob += token_score

    return total_log_prob

# Example experiment
A = "I have just stepped on a nail. Ouch,"
B = "it hurts!"

score = log_prob(A, B)

print(f"\nlog P(B | A) = {score:.4f}")


log P(B | A) = -4.5083
